# 🧹 Handling Missing Data
**One-line description:** Learn to detect, visualize, and impute missing values so your models never crash on NaNs.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/muhammadtalhaishtiaq/ai-orchestrator/blob/main/01-data-preprocessing/01_handling_missing_data.ipynb)


In [ ]:
# Install required libraries (run this cell first in Google Colab)
!pip install missingno scikit-learn pandas numpy matplotlib seaborn --quiet


## 📖 What is Missing Data?

Missing data occurs when no value is stored for a variable in an observation. In pandas, missing values appear as `NaN` (Not a Number) or `None`.

**Analogy:** Imagine filling out a survey — some respondents skip questions. If you try to calculate the average age but 20% of people left that field blank, your calculation will fail or be biased. That's the missing data problem in a nutshell.

### Types of Missingness:
| Type | Meaning | Example |
|------|---------|---------|
| **MCAR** (Missing Completely At Random) | Missingness unrelated to any data | Random sensor failure |
| **MAR** (Missing At Random) | Missingness related to *observed* data | Younger people skip income question |
| **MNAR** (Missing Not At Random) | Missingness related to the *missing value itself* | Very high earners skip salary question |


## 💡 Why Does It Matter?

- Most ML algorithms **cannot handle NaN values** — they throw errors
- Dropping all rows with missing data can **lose critical information**
- Naive imputation can **introduce bias** into your model
- Understanding the *mechanism* of missingness guides the right strategy


## ⚙️ How Does It Work?

The pipeline for handling missing data:
1. **Detect** — find where and how much data is missing
2. **Visualize** — understand patterns in missingness
3. **Decide** — deletion vs. imputation based on MCAR/MAR/MNAR
4. **Impute** — fill with mean/median/mode, KNN, or iterative methods
5. **Validate** — ensure imputation didn't distort distributions


## 🛠️ Hands-on Code

### Step 1: Create a Titanic-style synthetic dataset


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import missingno as msno
from sklearn.impute import SimpleImputer, KNNImputer, IterativeImputer
from sklearn.experimental import enable_iterative_imputer  # noqa
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)
n = 891  # Same size as real Titanic dataset

# Create a realistic Titanic-style dataset with deliberate missing patterns
df = pd.DataFrame({
    'PassengerId': range(1, n + 1),
    'Survived':    np.random.choice([0, 1], n, p=[0.62, 0.38]),
    'Pclass':      np.random.choice([1, 2, 3], n, p=[0.24, 0.21, 0.55]),
    'Sex':         np.random.choice(['male', 'female'], n, p=[0.65, 0.35]),
    'Age':         np.random.normal(29.7, 14.5, n).clip(0.5, 80),
    'SibSp':       np.random.choice([0,1,2,3,4], n, p=[0.68,0.23,0.06,0.02,0.01]),
    'Parch':       np.random.choice([0,1,2,3], n, p=[0.76,0.13,0.09,0.02]),
    'Fare':        np.random.exponential(32, n).clip(0, 512),
    'Embarked':    np.random.choice(['S','C','Q'], n, p=[0.72,0.19,0.09]),
    'Cabin':       np.random.choice(['A','B','C','D','E', None], n, p=[0.03,0.05,0.06,0.04,0.04,0.78]),
})

# Introduce realistic missing patterns:
# Age: MAR — older passengers in 3rd class more likely to have missing age
missing_age_mask = (df['Pclass'] == 3) & (np.random.rand(n) < 0.30)
missing_age_mask |= (np.random.rand(n) < 0.10)  # random 10% baseline
df.loc[missing_age_mask, 'Age'] = np.nan

# Fare: MCAR — completely random ~1%
df.loc[np.random.choice(n, 15, replace=False), 'Fare'] = np.nan

# Embarked: MCAR — very few missing
df.loc[np.random.choice(n, 3, replace=False), 'Embarked'] = np.nan

print("Dataset shape:", df.shape)
print("\nMissing value counts:")
print(df.isnull().sum())
print("\nMissing value percentages:")
print((df.isnull().sum() / len(df) * 100).round(2))


### Step 2: Visualize Missing Data Patterns


In [ ]:
# --- Visualization 1: Heatmap of missing values ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart of missing value counts
missing_counts = df.isnull().sum().sort_values(ascending=False)
missing_counts = missing_counts[missing_counts > 0]
axes[0].bar(missing_counts.index, missing_counts.values, color='coral', edgecolor='black')
axes[0].set_title('Missing Value Counts by Feature', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Number of Missing Values')
axes[0].set_xlabel('Feature')
for i, v in enumerate(missing_counts.values):
    axes[0].text(i, v + 1, str(v), ha='center', fontweight='bold')

# Heatmap of missingness
msno.heatmap(df, ax=axes[1], cmap='RdYlGn', fontsize=12)
axes[1].set_title('Missingness Correlation Heatmap', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('missing_data_overview.png', dpi=100, bbox_inches='tight')
plt.show()
print("Missingness heatmap: values close to 1 = these columns tend to be missing TOGETHER")


In [ ]:
# --- Visualization 2: Missingness matrix (missingno) ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Matrix plot using missingno
msno.matrix(df, ax=axes[0], sparkline=False, fontsize=10, color=(0.3, 0.5, 0.8))
axes[0].set_title('Missingness Matrix
(white = missing)', fontsize=13, fontweight='bold')

# Distribution of Age: missing vs. present
df_age_present = df[df['Age'].notna()]['Age']
df_age_missing_pclass = df[df['Age'].isna()]['Pclass']

axes[1].hist(df_age_present, bins=30, color='steelblue', alpha=0.7, label='Age Present')
axes[1].axvline(df_age_present.mean(), color='red', linestyle='--', label=f'Mean: {df_age_present.mean():.1f}')
axes[1].axvline(df_age_present.median(), color='orange', linestyle='--', label=f'Median: {df_age_present.median():.1f}')
axes[1].set_title('Distribution of Observed Age Values', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Age')
axes[1].set_ylabel('Count')
axes[1].legend()

plt.tight_layout()
plt.savefig('age_distribution.png', dpi=100, bbox_inches='tight')
plt.show()


### Step 3: Deletion Strategies


In [ ]:
# --- Listwise deletion: drop any row with at least one NaN ---
df_listwise = df.dropna()
print(f"Original shape:         {df.shape}")
print(f"After listwise deletion: {df_listwise.shape}")
print(f"Rows lost: {df.shape[0] - df_listwise.shape[0]} ({(df.shape[0]-df_listwise.shape[0])/df.shape[0]*100:.1f}%)")

print("\n--- WARNING ---")
print("Listwise deletion removes entire rows. If missingness is NOT MCAR,")
print("this can introduce significant bias into your analysis!")


In [ ]:
# --- Column deletion: drop columns with high missingness ---
threshold = 0.50  # drop columns where > 50% of values are missing
col_missing_pct = df.isnull().sum() / len(df)
cols_to_drop = col_missing_pct[col_missing_pct > threshold].index.tolist()
df_col_dropped = df.drop(columns=cols_to_drop)
print(f"Columns dropped (>{threshold*100:.0f}% missing): {cols_to_drop}")
print(f"Remaining columns: {df_col_dropped.columns.tolist()}")


### Step 4: Imputation Strategies


In [ ]:
# Select only numeric columns for imputation demonstration
numeric_cols = ['Age', 'Fare', 'SibSp', 'Parch', 'Pclass']
df_numeric = df[numeric_cols].copy()

print("Missing values before imputation:")
print(df_numeric.isnull().sum())
print()

# --- Strategy 1: Mean imputation ---
mean_imputer = SimpleImputer(strategy='mean')
df_mean = pd.DataFrame(mean_imputer.fit_transform(df_numeric),
                       columns=numeric_cols)
print("Strategy 1: Mean Imputation")
print(f"  Age imputed with mean: {df_numeric['Age'].mean():.2f}")

# --- Strategy 2: Median imputation ---
median_imputer = SimpleImputer(strategy='median')
df_median = pd.DataFrame(median_imputer.fit_transform(df_numeric),
                         columns=numeric_cols)
print(f"\nStrategy 2: Median Imputation")
print(f"  Age imputed with median: {df_numeric['Age'].median():.2f}")

# --- Strategy 3: KNN Imputation ---
knn_imputer = KNNImputer(n_neighbors=5)
df_knn = pd.DataFrame(knn_imputer.fit_transform(df_numeric),
                      columns=numeric_cols)
print(f"\nStrategy 3: KNN Imputation (k=5)")
print(f"  Uses 5 nearest neighbors to estimate missing values")

# --- Strategy 4: Iterative Imputation (MICE) ---
iter_imputer = IterativeImputer(max_iter=10, random_state=42)
df_iter = pd.DataFrame(iter_imputer.fit_transform(df_numeric),
                       columns=numeric_cols)
print(f"\nStrategy 4: Iterative Imputation (MICE)")
print(f"  Models each feature as a function of all others, iterates until convergence")


In [ ]:
# --- Visualization 3: Compare imputation methods on Age distribution ---
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

original_age = df['Age'].dropna()
methods = [
    (df_mean['Age'], 'Mean Imputation', 'coral'),
    (df_median['Age'], 'Median Imputation', 'steelblue'),
    (df_knn['Age'], 'KNN Imputation (k=5)', 'green'),
    (df_iter['Age'], 'Iterative (MICE)', 'purple'),
]

for ax, (imputed_age, title, color) in zip(axes.flat, methods):
    ax.hist(original_age, bins=30, alpha=0.5, label='Original (observed)', color='gray')
    ax.hist(imputed_age, bins=30, alpha=0.5, label=title, color=color)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel('Age')
    ax.set_ylabel('Count')
    ax.legend()

plt.suptitle('Comparison of Imputation Methods on Age Distribution', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('imputation_comparison.png', dpi=100, bbox_inches='tight')
plt.show()


In [ ]:
# --- Mode imputation for categorical columns ---
cat_imputer = SimpleImputer(strategy='most_frequent')
df_cat = df[['Embarked', 'Cabin']].copy()
print("Missing in categorical before imputation:")
print(df_cat.isnull().sum())

df_cat_imputed = pd.DataFrame(cat_imputer.fit_transform(df_cat),
                              columns=['Embarked', 'Cabin'])
print("\nMissing in categorical after mode imputation:")
print(df_cat_imputed.isnull().sum())
print(f"\nEmbarked mode: '{df['Embarked'].mode()[0]}'")


## 📌 Key Takeaways

- **Always visualize** missingness before deciding on a strategy — use `missingno`
- **MCAR** → any strategy works; **MAR** → imputation preferred; **MNAR** → requires domain knowledge
- **Mean imputation** is quick but distorts distribution variance
- **Median imputation** is robust to outliers — use for skewed features
- **KNN imputation** leverages similar samples — better quality, slower
- **Iterative (MICE) imputation** is gold standard for complex datasets
- **Never impute on test set using test set statistics** — always `fit` on train, `transform` on both
- Cabin-like columns with >70% missing are often best **dropped**
